# Retail Sales Performance Analysis
### Exploratory Data Analysis using Python (Pandas, Matplotlib, Seaborn)

**Goal:** Analyze retail sales data to uncover trends, identify the Q3 2023 revenue dip, and generate business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/sales_data.csv')
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df.head()

In [ ]:
df.info()
print('\nMissing values:\n', df.isnull().sum())
print('\nShape:', df.shape)

## 2. Data Cleaning

In [ ]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Check for negative/invalid values
df = df[(df['Quantity'] > 0) & (df['Price'] > 0) & (df['TotalAmount'] > 0)]

print('Cleaned dataset shape:', df.shape)

## 3. Key Business Metrics (KPIs)

In [ ]:
total_revenue = df['TotalAmount'].sum()
total_orders = df['OrderID'].nunique()
avg_order_value = df['TotalAmount'].mean()

print(f'Total Revenue: ₹{total_revenue:,.2f}')
print(f'Total Orders: {total_orders:,}')
print(f'Average Order Value: ₹{avg_order_value:,.2f}')

## 4. Monthly Revenue Trend

In [ ]:
monthly = df.groupby(df['OrderDate'].dt.to_period('M'))['TotalAmount'].sum()

plt.figure(figsize=(12,5))
monthly.plot(kind='line', marker='o', color='#4a4a8a')
plt.title('Monthly Revenue Trend (2023-2024)')
plt.xlabel('Month')
plt.ylabel('Revenue (₹)')
plt.tight_layout()
plt.savefig('../images/monthly_trend.png', dpi=150)
plt.show()

## 5. Revenue by Region

In [ ]:
region_rev = df.groupby('Region')['TotalAmount'].sum().sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=region_rev.index, y=region_rev.values, palette='viridis')
plt.title('Total Revenue by Region')
plt.ylabel('Revenue (₹)')
plt.tight_layout()
plt.savefig('../images/region_revenue.png', dpi=150)
plt.show()

## 6. Category-wise Revenue Share

In [ ]:
cat_rev = df.groupby('Category')['TotalAmount'].sum().sort_values(ascending=False)

plt.figure(figsize=(7,7))
plt.pie(cat_rev, labels=cat_rev.index, autopct='%1.1f%%', startangle=90,
        colors=sns.color_palette('Set2'))
plt.title('Revenue Share by Category')
plt.tight_layout()
plt.savefig('../images/category_share.png', dpi=150)
plt.show()

## 7. Quarterly Trend — Detecting the Q3 2023 Dip

In [ ]:
df['Year'] = df['OrderDate'].dt.year
df['Quarter'] = df['OrderDate'].dt.quarter

quarterly = df.groupby(['Year','Quarter'])['TotalAmount'].sum().reset_index()
quarterly['Period'] = quarterly['Year'].astype(str) + '-Q' + quarterly['Quarter'].astype(str)

plt.figure(figsize=(12,5))
sns.barplot(x='Period', y='TotalAmount', data=quarterly, color='#2d6a4f')
plt.title('Quarterly Revenue Trend')
plt.xticks(rotation=45)
plt.ylabel('Revenue (₹)')
plt.tight_layout()
plt.savefig('../images/quarterly_trend.png', dpi=150)
plt.show()

print(quarterly)

## 8. Customer Segment Analysis

In [ ]:
segment_analysis = df.groupby('CustomerSegment').agg(
    Orders=('OrderID','count'),
    Revenue=('TotalAmount','sum'),
    AvgRating=('Rating','mean')
).sort_values('Revenue', ascending=False)

segment_analysis

## 9. Top 10 Products by Revenue

In [ ]:
top_products = df.groupby('Product')['TotalAmount'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(x=top_products.values, y=top_products.index, palette='mako')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Revenue (₹)')
plt.tight_layout()
plt.savefig('../images/top_products.png', dpi=150)
plt.show()

## 10. Key Insights Summary

- 📉 **Q3 2023 showed a clear revenue dip** — correlating with inventory shortages
- 🏆 **Electronics** is the top revenue-generating category
- 🌍 Regional performance varies significantly — opportunity to reallocate marketing budget
- 👥 **Loyal customers** contribute a disproportionately high share of revenue despite fewer orders
- 💳 UPI and Credit Card are the most preferred payment modes

## Recommendations
1. Strengthen inventory planning ahead of Q3 to avoid stockouts
2. Invest more marketing budget in top-performing regions
3. Launch loyalty programs to convert 'New' customers into 'Returning/Loyal'
4. Bundle low-rated products with top sellers to boost their visibility